In [1]:
import numpy as np
import pandas as pd
import requests
import duckdb
import os
import s3fs
np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

FILE_KEY_S3 = "bistrots_sirene.csv"
FILE_PATH_S3_bistrots = BUCKET + "/" + FILE_KEY_S3

FILE_KEY_S3 = "filosofi.csv"
FILE_PATH_S3_filosofi = BUCKET + "/" + FILE_KEY_S3

In [ ]:
# Data prep Sirene ----------------------------
## Modifie le format de la dateDernierTraitementEtablissement (enlever l'heure) et de la dateCreationEtablissement 
## Met à NA les dates de création au 1er janvier 1900 ou 1901
bistrots_sirene["date_dernier_traitement"] = pd.to_datetime(bistrots_sirene.dateDernierTraitementEtablissement).dt.normalize()
bistrots_sirene["date_creation"] = pd.to_datetime(bistrots_sirene.dateCreationEtablissement.replace("1900-01-01", np.nan).replace("1901-01-01", np.nan)).dt.normalize()
bistrots_sirene.head()

In [ ]:
#Agrégation Paris-Lyon-Marseille.

arr = bistrots_sirene["codeCommuneEtablissement"].astype(str)
bistrots_sirene["code_commune"] = arr.mask(arr.str.startswith("751"), "75056") \
                                     .mask(arr.str.match(r"^6938[1-9]$"), "69123") \
                                     .mask(arr.str.match(r"^132(0[1-9]|1[0-6])$"), "13055")
#Paris
(bistrots_sirene[bistrots_sirene.libelleCommuneEtablissement.str.contains("PARIS")]["code_commune"]=='75056').describe()
#Lyon
(bistrots_sirene[bistrots_sirene.libelleCommuneEtablissement.str.contains("LYON")]["code_commune"]=='69123').describe()
#Marseille
(bistrots_sirene[bistrots_sirene.libelleCommuneEtablissement.str.contains("MARSEILLE")]["code_commune"]=='13055').describe()
bistrots_par_ville = (
    bistrots_sirene
    .assign(
        age=(
            pd.Timestamp("2026-03-01")
            - pd.to_datetime(
                bistrots_sirene["dateCreationEtablissement"],
                errors="coerce"
            )
        ).dt.days / 365.25
    )
    .groupby(["code_commune"])
    .agg(
        nb_etablissements=("siret", "nunique"),
        age_moyen_bistrots=("age", "mean")
    )
    .reset_index()
    .sort_values("nb_etablissements", ascending=False)
)